In [2]:
import os

In [3]:
%pwd

'c:\\Users\\jyoti\\OneDrive\\Desktop\\ChickenDC\\Chicken_Disease_Classification\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\jyoti\\OneDrive\\Desktop\\ChickenDC\\Chicken_Disease_Classification'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [8]:
from CNNClassifier.constants import *
from CNNClassifier.utils.common import read_yaml, create_directories
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [9]:
import os
import ssl
import shutil
import urllib.request as request
import zipfile
from pathlib import Path

from CNNClassifier import logger
from CNNClassifier.utils.common import get_size

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        local_file = Path(self.config.local_data_file)

        if not os.path.exists(local_file):
            local_file.parent.mkdir(parents=True, exist_ok=True)

            context = ssl._create_unverified_context()
            with request.urlopen(self.config.source_URL, context=context) as response:
                with open(local_file, "wb") as f:
                    shutil.copyfileobj(response, f)

            logger.info(f"Downloaded file to {local_file}")
        else:
            logger.info(f"File already exists of size: {get_size(local_file)}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-16 17:31:40,565 : INFO : common : yaml file: config\config.yaml loaded successfully]
[2026-07-16 17:31:40,567 : INFO : common : yaml file: params.yaml loaded successfully]
[2026-07-16 17:31:40,567 : INFO : common : created directory at: artifacts]
[2026-07-16 17:31:40,568 : INFO : common : created directory at: artifacts/data_ingestion]
[2026-07-16 17:31:44,012 : INFO : 2895208032 : Downloaded file to artifacts\data_ingestion\data.zip]
